# Customer Product Review – Sentiment Analysis
**Goal:** classify product reviews as *positive* or *negative*.

| Model | Idea |
|---|---|
| **Baseline** | Bag-of-Words + Naive Bayes |
| **Improved** | TF-IDF (1-2 grams) + Logistic Regression, tuned with GridSearchCV |

**Data source:** Amazon Polarity reviews (Hugging Face `amazon_polarity`, open, no login).
Fallback: UCI *Sentiment Labelled Sentences* (Amazon subset).

In [ ]:
# !pip install -q datasets scikit-learn pandas matplotlib seaborn joblib   # uncomment on Colab
import re, warnings, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

## 1. Load data

In [ ]:
N_SAMPLES = 40_000   # keep small so it runs fast; raise for better scores

def load_data():
    # Option A: Hugging Face (3.6M Amazon reviews, we sample)
    try:
        from datasets import load_dataset
        ds = load_dataset("amazon_polarity", split="train").shuffle(seed=RANDOM_STATE).select(range(N_SAMPLES))
        df = ds.to_pandas()
        df["review"] = df["title"].fillna("") + ". " + df["content"].fillna("")
        return df[["review", "label"]], "amazon_polarity (Hugging Face)"
    except Exception as e:
        print("HF load failed ->", type(e).__name__, "| trying UCI fallback")
    # Option B: UCI Sentiment Labelled Sentences (1000 Amazon reviews)
    import io, zipfile, urllib.request
    url = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
    name = [n for n in z.namelist() if n.endswith("amazon_cells_labelled.txt")][0]
    df = pd.read_csv(z.open(name), sep="\t", header=None, names=["review", "label"])
    return df, "UCI amazon_cells_labelled"

df, source = load_data()
print("Source:", source, "| rows:", len(df))
df.head()

## 2. Quick EDA

In [ ]:
df = df.dropna().drop_duplicates(subset="review").reset_index(drop=True)
df["n_words"] = df["review"].str.split().str.len()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.countplot(x="label", data=df, ax=ax[0]); ax[0].set_title("Class balance (0=neg, 1=pos)")
sns.histplot(df["n_words"].clip(upper=300), bins=40, ax=ax[1]); ax[1].set_title("Review length (words)")
plt.tight_layout(); plt.show()
print(df["label"].value_counts(normalize=True).round(3))

## 3. Clean text & split

In [ ]:
def clean(t):
    t = t.lower()
    t = re.sub(r"<.*?>", " ", t)          # html tags
    t = re.sub(r"http\S+", " ", t)         # urls
    t = re.sub(r"[^a-z' ]", " ", t)       # keep letters
    return re.sub(r"\s+", " ", t).strip()

df["clean"] = df["review"].apply(clean)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, stratify=df["label"], random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

## 4. Baseline model – Bag-of-Words + Naive Bayes

In [ ]:
baseline = Pipeline([
    ("bow", CountVectorizer(max_features=5000)),
    ("clf", MultinomialNB()),
])
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)
print("Baseline accuracy:", round(accuracy_score(y_test, pred_base), 4))
print(classification_report(y_test, pred_base, target_names=["negative", "positive"]))

## 5. Improved model – TF-IDF (1-2 grams) + Logistic Regression
Why better: bigrams capture phrases like *"not good"*, TF-IDF down-weights common words,
Logistic Regression learns per-word weights, and GridSearchCV tunes the settings.

In [ ]:
improved = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words=None)),
    ("clf", LogisticRegression(max_iter=1000, solver="liblinear")),
])
grid = {
    "tfidf__max_features": [50_000, 100_000],
    "clf__C": [1, 3, 10],
}
search = GridSearchCV(improved, grid, cv=3, scoring="f1", n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
best = search.best_estimator_
print("Best params:", search.best_params_)
pred_best = best.predict(X_test)
print("Improved accuracy:", round(accuracy_score(y_test, pred_best), 4))
print(classification_report(y_test, pred_best, target_names=["negative", "positive"]))

## 6. Compare models

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline (BoW + NB)", "Improved (TF-IDF + LR, tuned)"],
    "Accuracy": [accuracy_score(y_test, pred_base), accuracy_score(y_test, pred_best)],
    "F1": [f1_score(y_test, pred_base), f1_score(y_test, pred_best)],
}).round(4)
display(results)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, p, t in zip(ax, [pred_base, pred_best], ["Baseline", "Improved"]):
    sns.heatmap(confusion_matrix(y_test, p), annot=True, fmt="d", cmap="Blues", ax=a,
                xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
    a.set_title(t); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout(); plt.show()

## 7. What did the model learn? (top words)

In [ ]:
vec, clf = best.named_steps["tfidf"], best.named_steps["clf"]
words = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
print("Most NEGATIVE:", ", ".join(words[order[:15]]))
print("Most POSITIVE:", ", ".join(words[order[-15:]][::-1]))

## 8. Save model & try it

In [ ]:
joblib.dump(best, "sentiment_model.joblib")

def predict_review(text):
    p = best.predict_proba([clean(text)])[0][1]
    return ("positive" if p >= 0.5 else "negative"), round(float(p), 3)

for r in ["Great quality, works perfectly, totally worth the price!",
          "Stopped working after two days. Waste of money.",
          "Not good at all, very disappointed"]:
    print(r, "->", predict_review(r))

## 9. Conclusion
- Baseline gives a quick reference score.
- The improved pipeline adds bigrams, TF-IDF weighting and tuning, and beats the baseline on both accuracy and F1.
- **Next steps:** try a Transformer (DistilBERT), handle neutral/3-star reviews, add a Streamlit demo.